In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json

import jax
import numpy as np
from interpax import Interpolator1D

# TODO: this setup should be done taking into account the device description as a submodule

# # Get sensor positions from device description json file
# with open("/home/zkeith/proj/ONW/ryan-onsim/Device-description/device_description/SPARC/240510/device.json") as f:
#     device_description = json.load(f)

# BpLowmnInd = np.where(np.array([x['name'][0:7] for x in device_description['magnetics']['b_field_pol_probe']]) == 'BP-LOMN')[0]
# BnLowmnInd = np.where(np.array([x['name'][0:7] for x in device_description['magnetics']['b_field_pol_probe']]) == 'BN-LOMN')[0]

# phi_probes = [device_description['magnetics']['b_field_pol_probe'][i]['position']['phi'] for i in BpLowmnInd]
# phi_sens = [device_description['magnetics']['b_field_pol_probe'][i]['position']['phi'] for i in BnLowmnInd]

# # Using 12 probe design from 2 years ago
# # 8, 211, 91, 271, 288, 171, 228, 351, 131, 48, 328, 311
# probe_order_indices = [0, 9, 3, 11, 12, 7, 10, 15, 5, 2, 14, 13]
# probe_connection_indices = []
# for i in range(len(probe_order_indices)):
#     if i != len(probe_order_indices) - 1:
#         probe_connection_indices.append([probe_order_indices[i], probe_order_indices[i+1]])
#     else:
#         probe_connection_indices.append([probe_order_indices[i], probe_order_indices[0]])

# probe_connections = []
# for connection in probe_connection_indices:
#     probe_connections.append([phi_probes[connection[0]], phi_probes[connection[1]]])

# Get sensor positions from lown_design.txt config file
filepath = "../../data/tearing/"
fname = "lown_design.txt"
# The text file has one tuple of (phi1, phi2) per line
with open(filepath + fname) as f:
    probe_connections = [tuple(map(float, line.split(", "))) for line in f]

# Get frequency responses
fname = "21_mode_resp_data.txt"
out = np.loadtxt(filepath + fname, skiprows=1)

freq = out[:,0]
Bp_per_A = out[:,1] # Bp (poloidal field) per Amp of tearing mode current
Br_per_A = out[:,2] # Br (radial field) per Amp of tearing mode current

# Make Jax-compatible interpolators to determine 
# measured field per Amp of tearing mode current at arbitrary rotation frequencies
jax.config.update("jax_platforms", "cpu")
func_Bp_per_A = Interpolator1D(freq, Bp_per_A)
func_Br_per_A = Interpolator1D(freq, Br_per_A)

In [ ]:
from popsim import param_utils
from popsim.modules.tearing import Island, Tearing, generate_disruption_phase_trajectory, generate_island_rotation_phase_trajectory

# Set up tearing module
dt = 1e-4 / 3  # s
time_base = param_utils.make_time_base(t0=0.0, t1=7.0, dt=dt)
tearing_config = Tearing.Config(
    magx_time=time_base
)

islands = [Island(3,2), Island(2,1)]

W = {island: 0.0 for island in islands}
F = {island: 0.0 for island in islands}
mode_phase={island: 0.0 for island in islands}

tearing_initial_state = Tearing.State(W=W, F=F, mode_phase=mode_phase)

rot_dur = 1.0
trigger_time = 5.5
disrupt_time = 6.5
dur_tq_to_spike = 1e-3
dur_cq = 10e-3
survival_time = 0.3
locking_dur = 0.2

tearing_params = Tearing.Params(
    rot_dur=1.0,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(
        disrupt_time, dur_tq_to_spike, time_base, dt
    ),
    island_rotation_phase=generate_island_rotation_phase_trajectory(
        trigger_time, rot_dur, locking_dur, time_base, dt
    ),
)

tearing_module = Tearing(config=tearing_config, islands=islands)

In [ ]:
import jax

from popsim.modules.magnetic_measurements import LowNArray
from popsim.simulate import simulate

lown_array_config = LowNArray.Config(
    probe_connections=probe_connections,
    reconstructed_modes=[1,2,3]
)

lown_array_initial_state = LowNArray.State(tearing_state=tearing_initial_state)

lown_array_params = LowNArray.Params(
    tearing_params=tearing_params,
)

lown_array_module = LowNArray(config=lown_array_config, tearing_module=tearing_module, func_Bp_per_A=func_Bp_per_A)

jax.config.update("jax_platforms", "cpu")
lown_xarray = simulate(lown_array_module, time_base, lown_array_initial_state, lown_array_params)

In [ ]:
from popsim.visualize import visualize_time_series

#lown_xarray.to_netcdf("lown_simulation.nc")
visualize_time_series(lown_xarray, max_cols=2)